# Version 2

In [21]:
from typing import List
import pandas as pd
from fedbiomed.common.dataset._nipoppy_dataset import NipoppyDataset
from fedbiomed.common.datamanager import DataManager
from fedbiomed.common.training_plans._base_training_plan import BaseTrainingPlan
from fedbiomed.common.training_plans import FedSGDRegressor

from fedbiomed.researcher.federated_workflows import Experiment
from fedbiomed.researcher.aggregators.fedavg import FedAverage


class RegressionTrainingPlan(FedSGDRegressor):

    # columns
    TERMURL_AGE = "nb:Age"
    TERMURL_SEX = "nb:Sex"
    TERMURL_COG_DECLINE = "fl:cognitive_decline_status"
    TERMURL_COG_DECLINE_AVAILABILITY = "fl:cognitive_decline_availability"
    TERMURL_DIAGNOSIS = "nb:Diagnosis"

    # values
    TERMURL_AVAILABLE = "nb:available"
    TERMURL_UNAVAILABLE = "nb:unavailable"
    TERMURL_MALE = "snomed:248153007"
    TERMURL_FEMALE = "snomed:248152002"
    TERMURL_HEALTHY_CONTROL = "ncit:C94342"

    # for derivatives specs
    FS_NAME = "freesurfer"
    FS_VERSION = "7.3.2"
    FS_STATS_NAME = "fs_stats"
    FS_STATS_VERSION = "0.2.1"
    SUFFIX_APARC = "-aparc.DKTatlas-thickness.tsv"
    SUFFIX_ASEG = "-aseg-volume.tsv"

    dataset = None

    def transform(self, data: pd.Series) -> pd.Series:
        data = self._transform_aseg(data)
        data = self._transform_map(data)
        return data

    def _transform_aseg(self, data: pd.Series) -> pd.Series:
        data = data.drop(
            index=[
                "3rd-Ventricle",
                "4th-Ventricle",
                "5th-Ventricle",
                "Brain-Stem",
                "BrainSegVol",
                "BrainSegVol-to-eTIV",
                "BrainSegVolNotVent",
                "CC_Anterior",
                "CC_Central",
                "CC_Mid_Anterior",
                "CC_Mid_Posterior",
                "CC_Posterior",
                "CSF",
                "CerebralWhiteMatterVol",
                "CortexVol",
                "Left-Cerebellum-Cortex",
                "Left-Cerebellum-White-Matter",
                "Left-Inf-Lat-Vent",
                "Left-VentralDC",
                "Left-WM-hypointensities",
                "Left-choroid-plexus",
                "Left-non-WM-hypointensities",
                "Left-vessel",
                "MaskVol",
                "MaskVol-to-eTIV",
                "Optic-Chiasm",
                "Right-Cerebellum-Cortex",
                "Right-Cerebellum-White-Matter",
                "Right-Inf-Lat-Vent",
                "Right-VentralDC",
                "Right-WM-hypointensities",
                "Right-choroid-plexus",
                "Right-non-WM-hypointensities",
                "Right-vessel",
                "SubCortGrayVol",
                "SupraTentorialVol",
                "SupraTentorialVolNotVent",
                "SurfaceHoles",
                "TotalGrayVol",
                "WM-hypointensities",
                "lhCerebralWhiteMatterVol",
                "lhCortexVol",
                "lhSurfaceHoles",
                "non-WM-hypointensities",
                "rhCerebralWhiteMatterVol",
                "rhCortexVol",
                "rhSurfaceHoles",
            ]
        )
        return data

    def _transform_map(self, data: pd.Series) -> pd.Series:
        if self.TERMURL_SEX in data.index:
            data[self.TERMURL_SEX] = {self.TERMURL_FEMALE: 0, self.TERMURL_MALE: 1}[
                data[self.TERMURL_SEX]
            ]
        return data

        # if self.TERMURL_COG_DECLINE in df.columns:
        #     specific_transformers.append(
        #         (
        #             OneHotEncoder(
        #                 drop=[self.TERMURL_UNAVAILABLE],
        #                 sparse_output=False,
        #                 feature_name_combiner=lambda x, _: x,
        #             ),
        #             [self.TERMURL_COG_DECLINE],
        #         )
        #     )

        # table_vectorizer = TableVectorizer(specific_transformers=specific_transformers)
        # df = table_vectorizer.fit_transform(df)
        # return df.squeeze().T

    # def _preprocess_select_hc(self, df: pd.DataFrame) -> pd.DataFrame:
    #     df = df.loc[df[self.TERMURL_DIAGNOSIS] == self.TERMURL_HEALTHY_CONTROL]
    #     df = df.drop(columns=[self.TERMURL_DIAGNOSIS])
    #     return df

    # def _preprocess_dropna(self, df: pd.DataFrame) -> pd.DataFrame:
    #     return df.dropna(axis="index", how="any")

    def init_dependencies(self: BaseTrainingPlan) -> List[str]:
        deps = [
            "from typing import List",
            "import pandas as pd",
            "from fedbiomed.common.dataset._nipoppy_dataset import NipoppyDataset",
            "from fedbiomed.common.datamanager import DataManager",
            "from fedbiomed.common.training_plans._base_training_plan import BaseTrainingPlan",
            "from fedbiomed.common.training_plans import FedSGDRegressor",
        ]
        return deps

    def training_data(self: BaseTrainingPlan) -> DataManager:
        model_args = self.model_args() or {}
        dataset = NipoppyDataset(
            target=self.TERMURL_AGE,
            phenotypes=[self.TERMURL_AGE, self.TERMURL_SEX],
            derivatives=[
                (
                    self.FS_NAME,
                    self.FS_VERSION,
                    f"idp/{self.FS_STATS_NAME}-{self.FS_STATS_VERSION}/fs{self.FS_VERSION}{self.SUFFIX_ASEG}",
                )
            ],
            transform=self.transform,
        )
        self.dataset = dataset
        return DataManager(dataset=dataset, shuffle=model_args.get("shuffle", False))


experiment = Experiment(
    tags=["test"],
    training_plan_class=RegressionTrainingPlan,
    model_args={
        # fedbiomed
        "eta0": 0.05,
        "random_state": 1,
        "n_features": 18,
        "n_classes": 2,  # ignored in regression tasks?
        # model
        "learning_rate": "constant",
        "penalty": "l2",
        # data-loading
        "shuffle": True,
    },
    round_limit=1,
    training_args={
        "num_updates": 1,
        "loader_args": {"batch_size": 50},
    },
    aggregator=FedAverage(),
    node_selection_strategy=None,
)

experiment.run()

2026-02-11 08:40:02,080 fedbiomed INFO - Updating training data. This action will update FederatedDataset, and the nodes that will participate to the experiment.

2026-02-11 08:40:02,086 fedbiomed DEBUG - Node: NODE_df747aff-2de0-420d-9287-2b285dfd389b polling for the tasks

2026-02-11 08:40:02,087 fedbiomed INFO - Node selected for training -> Default Node Name
Node ID is -> NODE_df747aff-2de0-420d-9287-2b285dfd389b

2026-02-11 08:40:02,190 fedbiomed DEBUG - Model file has been saved: /data/origami/michelle/projects/fedbiomed/fbm-researcher/var/experiments/Experiment_0024/model_18cba6df-cfcb-4f86-8c89-e4f6fc29af83.py

2026-02-11 08:40:02,259 fedbiomed WARNING - Option share_persistent_buffers is not supported in SKLearnTrainingPlan, it will be ignored.

...

2026-02-11 08:40:02,266 fedbiomed INFO - Sending request 
					 To: NODE_df747aff-2de0-420d-9287-2b285dfd389b 
					 Request: : TRAIN
 -----------------------------------------------------------------

2026-02-11 08:40:02,329 fedbiomed DEBUG - Node: NODE_df747aff-2de0-420d-9287-2b285dfd389b polling for the tasks

2026-02-11 08:40:02,488 fedbiomed INFO - TRAINING 
					 NODE_ID: NODE_df747aff-2de0-420d-9287-2b285dfd389b 
					 Node Name: Default Node Name 
					 Round 1 | Iteration: 1/1 (100%) | Samples: 50/50
 					 Loss squared_error: 2132.969971 
					 ---------

2026-02-11 08:40:02,637 fedbiomed INFO - Nodes that successfully reply in round 0 ['NODE_df747aff-2de0-420d-9287-2b285dfd389b']

1

In [19]:
from fedbiomed.common.dataset_types import DataReturnFormat

training_plan = RegressionTrainingPlan()
training_plan.training_data()
dataset = training_plan.dataset
dataset.complete_initialization(
    controller_kwargs={"root": "./example_dataset/my_dataset"},
    to_format=DataReturnFormat.SKLEARN,
)
print(dataset[2])

02/11/2026 08:36:54:DEBUG:Loading manifest from example_dataset/my_dataset/manifest.tsv


(array([0, 1.6421601269500776, -0.6502859445071476, 1.627589166597494,
       -0.5856936953375567, 0.0817167003608864, -0.8174513395714376,
       0.2381957448703632, -0.0885050366035554, 0.8665798234085362,
       -1.3439678262752042, 1.285511286548568, 1.2977346737804971,
       -0.5322431935575963, 1.0374046058699675, 1.4682311715954466,
       -0.5388355035053808, 0.945901085087668], dtype=object), array([76]))


# Version 1

In [ ]:
from typing import List
import pandas as pd
from fedbiomed.common.dataset._nipoppy_dataset import NipoppyDataset
from fedbiomed.common.datamanager import DataManager
from fedbiomed.common.training_plans._base_training_plan import BaseTrainingPlan
from fedbiomed.common.training_plans import FedSGDRegressor
from sklearn.preprocessing import OneHotEncoder
from skrub import TableVectorizer

from fedbiomed.researcher.federated_workflows import Experiment
from fedbiomed.researcher.aggregators.fedavg import FedAverage


class RegressionTrainingPlan(FedSGDRegressor):

    TERMURL_AGE = "nb:Age"
    TERMURL_SEX = "nb:Sex"
    TERMURL_COG_DECLINE = "fl:cognitive_decline_status"
    TERMURL_COG_DECLINE_AVAILABILITY = "fl:cognitive_decline_availability"
    TERMURL_DIAGNOSIS = "nb:Diagnosis"
    TERMURL_HEALTHY_CONTROL = "ncit:C94342"

    TERMURL_AVAILABLE = "nb:available"
    TERMURL_UNAVAILABLE = "nb:unavailable"
    TERMURL_MALE = "snomed:248153007"
    TERMURL_FEMALE = "snomed:248152002"
    TERMURL_HEALTHY_CONTROL = "ncit:C94342"

    # for derivatives specs
    FS_NAME = "freesurfer"
    FS_VERSION = "7.3.2"
    FS_STATS_NAME = "fs_stats"
    FS_STATS_VERSION = "0.2.1"
    SUFFIX_APARC = "-aparc.DKTatlas-thickness.tsv"
    SUFFIX_ASEG = "-aseg-volume.tsv"

    def transform_aseg(self, df: pd.DataFrame) -> pd.DataFrame:
        df = df.drop(
            columns=[
                "3rd-Ventricle",
                "4th-Ventricle",
                "5th-Ventricle",
                "Brain-Stem",
                "BrainSegVol",
                "BrainSegVol-to-eTIV",
                "BrainSegVolNotVent",
                "CC_Anterior",
                "CC_Central",
                "CC_Mid_Anterior",
                "CC_Mid_Posterior",
                "CC_Posterior",
                "CSF",
                "CerebralWhiteMatterVol",
                "CortexVol",
                "Left-Cerebellum-Cortex",
                "Left-Cerebellum-White-Matter",
                "Left-Inf-Lat-Vent",
                "Left-VentralDC",
                "Left-WM-hypointensities",
                "Left-choroid-plexus",
                "Left-non-WM-hypointensities",
                "Left-vessel",
                "MaskVol",
                "MaskVol-to-eTIV",
                "Optic-Chiasm",
                "Right-Cerebellum-Cortex",
                "Right-Cerebellum-White-Matter",
                "Right-Inf-Lat-Vent",
                "Right-VentralDC",
                "Right-WM-hypointensities",
                "Right-choroid-plexus",
                "Right-non-WM-hypointensities",
                "Right-vessel",
                "SubCortGrayVol",
                "SupraTentorialVol",
                "SupraTentorialVolNotVent",
                "SurfaceHoles",
                "TotalGrayVol",
                "WM-hypointensities",
                "lhCerebralWhiteMatterVol",
                "lhCortexVol",
                "lhSurfaceHoles",
                "non-WM-hypointensities",
                "rhCerebralWhiteMatterVol",
                "rhCortexVol",
                "rhSurfaceHoles",
            ]
        )
        return df

    def transform_select_hc(self, df: pd.DataFrame) -> pd.DataFrame:
        df = df.loc[df[self.TERMURL_DIAGNOSIS] == self.TERMURL_HEALTHY_CONTROL]
        df = df.drop(columns=[self.TERMURL_DIAGNOSIS])
        return df

    def transform_dropna(self, df: pd.DataFrame) -> pd.DataFrame:
        return df.dropna(axis="index", how="any")

    def transform_skrub(self, df: pd.DataFrame) -> pd.DataFrame:
        specific_transformers = []
        if self.TERMURL_SEX in df.columns:
            specific_transformers.append(
                (
                    OneHotEncoder(drop=[self.TERMURL_MALE], sparse_output=False),
                    [self.TERMURL_SEX],
                )
            )
        if self.TERMURL_COG_DECLINE in df.columns:
            specific_transformers.append(
                (
                    OneHotEncoder(
                        drop=[self.TERMURL_UNAVAILABLE],
                        sparse_output=False,
                        feature_name_combiner=lambda x, _: x,
                    ),
                    [self.TERMURL_COG_DECLINE],
                )
            )

        table_vectorizer = TableVectorizer(specific_transformers=specific_transformers)
        df = table_vectorizer.fit_transform(df)
        return df

    def init_dependencies(self: BaseTrainingPlan) -> List[str]:
        deps = [
            "import json",
            "import warnings",
            "from functools import cached_property",
            "from typing import List, Tuple, Type",
            "import numpy as np",
            "import pandas as pd",
            "from fedbiomed.common.dataset._nipoppy_dataset import NipoppyDataset",
            "from fedbiomed.common.datamanager import DataManager",
            "from fedbiomed.common.training_plans._base_training_plan import BaseTrainingPlan",
            "from fedbiomed.common.training_plans import FedSGDClassifier, FedSGDRegressor",
            "from nipoppy import NipoppyDataRetriever",
            "from sklearn.model_selection import StratifiedKFold",
            "from sklearn.preprocessing import OneHotEncoder",
            "from skrub import TableVectorizer",
        ]
        return deps

    def training_data(self: BaseTrainingPlan) -> DataManager:
        model_args = self.model_args()
        dataset = NipoppyDataset(
            phenotypes=[self.TERMURL_AGE, self.TERMURL_SEX, self.TERMURL_DIAGNOSIS],
            derivatives=[
                (
                    self.FS_NAME,
                    self.FS_VERSION,
                    f"idp/{self.FS_STATS_NAME}-{self.FS_STATS_VERSION}/fs{self.FS_VERSION}{self.SUFFIX_ASEG}",
                )
            ],
            transforms=[
                self.transform_aseg,
                self.transform_select_hc,
                self.transform_dropna,
                self.transform_skrub,
            ],
            n_splits=10,
            i_split=0,
            target=self.TERMURL_AGE,
            random_state=1,
            fname_stats=None,
            train=True,
            null=False,
        )
        return DataManager(dataset=dataset, shuffle=model_args.get("shuffle", False))


experiment = Experiment(
    tags=["test"],
    training_plan_class=RegressionTrainingPlan,
    model_args={
        # fedbiomed
        "eta0": 0.05,
        "random_state": 1,
        "n_features": 18,
        "n_classes": 2,  # ignored in regression tasks?
        # model
        "learning_rate": "constant",
        "penalty": "l2",
        # data-loading
        "shuffle": True,
    },
    round_limit=1,
    training_args={
        "num_updates": 1,
        "loader_args": {"batch_size": 50},
    },
    aggregator=FedAverage(),
    node_selection_strategy=None,
)

experiment.run()